# Copyjoe LangChain Quality Checks

이 노트북은 다음을 점검합니다.
- FastAPI 상태: `/health`, `/docs`, `/openapi.json`
- structured output 형식
- 업로드 -> RAG 인덱싱 -> 검색 품질
- 랜딩페이지 h1/h2/CTA/본문 추출


In [ ]:
import json
import os
from pathlib import Path

import requests
from docx import Document

BASE_URL = os.getenv("COPYJOE_BASE_URL", "http://127.0.0.1:8000")
session = requests.Session()


In [ ]:
health = session.get(f"{BASE_URL}/health")
docs = session.get(f"{BASE_URL}/docs")
openapi = session.get(f"{BASE_URL}/openapi.json")

print("health:", health.status_code, health.json())
print("docs:", docs.status_code)
print("openapi:", openapi.status_code, "paths=", len(openapi.json().get("paths", {})))


In [ ]:
payload = {
    "product_name": "Copyjoe",
    "target_audience": "퍼포먼스 마케터",
    "pain_point": "매번 새 카피를 만드는 데 시간이 오래 걸린다",
    "differentiator": "RAG + 웹근거 기반",
    "tone": "신뢰형",
    "objective": "click",
    "styles": ["head", "body", "cta", "slogan", "sns", "description"],
    "channel": "상세페이지",
    "language": "ko",
    "web_search_mode": False,
    "use_rag": False,
    "top_k": 5
}

res = session.post(f"{BASE_URL}/api/v1/copy/generate", json=payload)
res.raise_for_status()
copy_result = res.json()
print(json.dumps(copy_result, ensure_ascii=False, indent=2))


In [ ]:
required_keys = [
    "head", "body", "cta", "slogan", "sns", "description",
    "storyboard_outline", "rationale", "sources"
]
missing = [k for k in required_keys if k not in copy_result]
assert not missing, f"Missing keys: {missing}"
assert isinstance(copy_result["storyboard_outline"], list)
print("Structured output check: PASS")


In [ ]:
sample_docx = Path("tmp_quality_sample.docx")
doc = Document()
doc.add_heading("Copyjoe Product Notes", level=1)
doc.add_paragraph("Copyjoe는 마케팅 카피 자동 생성 도구입니다.")
doc.add_paragraph("핵심 고객은 퍼포먼스 마케터, AE, 1인 사업자입니다.")
doc.add_paragraph("차별점은 RAG와 Tavily 기반 최신 정보 반영입니다.")
doc.save(sample_docx)

with sample_docx.open("rb") as f:
    files = [("files", (sample_docx.name, f, "application/vnd.openxmlformats-officedocument.wordprocessingml.document"))]
    upload_res = session.post(f"{BASE_URL}/api/v1/files/upload", files=files)

print(upload_res.status_code, upload_res.json())
upload_body = upload_res.json()
doc_ids = [item["document_id"] for item in upload_body["files"] if item.get("success")]
assert doc_ids, "No documents uploaded successfully"


In [ ]:
index_payload = {"document_ids": doc_ids, "chunk_size": 500, "chunk_overlap": 80}
index_res = session.post(f"{BASE_URL}/api/v1/rag/index", json=index_payload)
print(index_res.status_code, index_res.json())

search_payload = {"query": "Copyjoe의 차별점은 무엇인가?", "top_k": 3}
search_res = session.post(f"{BASE_URL}/api/v1/rag/search", json=search_payload)
search_res.raise_for_status()
search_body = search_res.json()
print(json.dumps(search_body, ensure_ascii=False, indent=2))
assert len(search_body["results"]) > 0, "RAG search returned no results"


In [ ]:
payload["use_rag"] = True
payload["web_search_mode"] = False
rag_copy_res = session.post(f"{BASE_URL}/api/v1/copy/generate", json=payload)
rag_copy_res.raise_for_status()
rag_copy = rag_copy_res.json()
print("sources count:", len(rag_copy.get("sources", [])))
print("rationale:", rag_copy.get("rationale"))


In [ ]:
landing_payload = {"url": "https://example.com"}
landing_res = session.post(f"{BASE_URL}/api/v1/web/landing/analyze", json=landing_payload)
landing_res.raise_for_status()
landing = landing_res.json()
print(json.dumps({k: landing[k] for k in ["url", "title", "h1", "h2", "cta_buttons"]}, ensure_ascii=False, indent=2))
print("body length:", len(landing.get("body", "")))


In [ ]:
filled_fields = ["head", "body", "cta", "slogan", "sns", "description"]
non_empty = sum(1 for key in filled_fields if (copy_result.get(key) or "").strip())
storyboard_ok = len(copy_result.get("storyboard_outline", [])) >= 4
quality_score = round((non_empty / len(filled_fields)) * 60 + (20 if storyboard_ok else 0) + 20, 2)
print({
    "non_empty_fields": non_empty,
    "storyboard_ok": storyboard_ok,
    "quality_score": quality_score
})

Path("tmp_quality_sample.docx").unlink(missing_ok=True)
print("cleanup: tmp_quality_sample.docx removed")
